## 빠른 시작

### bind_tools와 StateGraph를 사용한 방식을 create_agent를 사용하여 한방에 해결~

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_tavily import TavilySearch
from langgraph.checkpoint.memory import InMemorySaver

# 1. 모델 초기화
model = init_chat_model("openai:gpt-4.1-mini", temperature=0)

# 2. 도구 정의
tavily = TavilySearch(max_results=3)

@tool
def calculator(expression: str) -> str:
    """수학 계산을 수행합니다. 사칙연산, 거듭제곱, 괄호를 지원합니다.

    Args:
        expression: 계산할 수학 표현식
    """
    import re
    if not re.match(r'^[0-9+\-*/().%\s]+$', expression):
        return "오류: 허용되지 않는 문자가 포함되어 있습니다."
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return f"{expression} = {result}"
    except Exception as e:
        return f"계산 오류: {str(e)}"

@tool
def get_current_datetime() -> str:
    """현재 날짜와 시간을 반환합니다."""
    from datetime import datetime
    return datetime.now().strftime("%Y년 %m월 %d일 %H시 %M분 %S초 (%A)")

# 3. 에이전트 생성
agent = create_agent(
    model=model,
    tools=[tavily, calculator, get_current_datetime],
    system_prompt="당신은 도움이 되는 AI 어시스턴트입니다.",
    checkpointer=InMemorySaver(),
)

# 4. 실행
config = {"configurable": {"thread_id": "session-1"}}
result = agent.invoke(
    {"messages": [{"role": "user", "content": "15 * 32는 얼마인가요?"}]},
    config=config,
)
print(result["messages"][-1].content)
